# 30. 전체 결과 비교

각 노트북이 `workspace/` 에 저장한 `*_ranking_metrics_*.json` 을 자동으로 찾아 하나의 표와 그래프로 비교한다.
임계값과 무관한 지표(PR-AUC, Recall@K, Recall@3K, MRR)와 임계값 기반 지표(Precision/Recall/F1)를 함께 산출하고,
원본 DB 와 마스킹 DB 의 결과가 같은 정답 공간에서 평가됐는지 검증한다.

# 4-모델 통합 비교 (CRIM / Valentine / DeepJoin / EmbDI)

각 모델 노트북에서 산출된 `*_ranking_metrics_*.json`을 자동 탐지·통합하여
같은 표에서 임계값-독립 지표(**PR-AUC**, **Recall@K**)를 비교한다.

## 입력 파일 (각 노트북 마지막 셀이 생성)
| 모델 | 경로 패턴 |
|---|---|
| CRIM (single) | `source/crim_ranking_metrics_{db}.json` |
| CRIM (ablation) | `source/ablation_ranking_metrics_0517_v34_final_ablation.json` |
| Valentine | `source/valentine_ranking_metrics_{db}.json` |
| DeepJoin | `source/output_deepjoin/deepjoin_ranking_metrics.json` |
| EmbDI | `source/embdi_ranking_metrics_{db}.json` |

## 공통 지표
- **PR-AUC** — 클래스 불균형에 강한 ranking 지표
- **Recall@K (K=|GT|)** — Valentine SIGMOD 2021 표준
- (보조) **Recall@3K**

매칭 키는 모두 `frozenset({(table_a, col_a), (table_b, col_b)})` 기반 undirected — 모델 간 비교가 공정함.

In [ ]:
# === Imports ===
import os
import json
import glob
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# ── 한글 폰트 자동 설정 (macOS 기준, 없으면 영어 fallback) ──
import matplotlib.font_manager as fm
_KOR_CANDIDATES = ['Apple SD Gothic Neo', 'AppleGothic', 'Nanum Gothic',
                   'Malgun Gothic', 'NanumGothic', 'Noto Sans CJK KR']
_available = {f.name for f in fm.fontManager.ttflist}
_selected = next((f for f in _KOR_CANDIDATES if f in _available), None)
if _selected:
    mpl.rcParams['font.family'] = _selected
    mpl.rcParams['axes.unicode_minus'] = False   # 마이너스 깨짐 방지
    print(f"🔤 한글 폰트 설정: '{_selected}'")
else:
    print("⚠ 한글 폰트 미발견 — 차트 한글이 □ 로 보일 수 있음")

# 경고 억제 (필요 시) — 폰트 missing glyph 경고 등
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

SOURCE_DIR  = Path('../workspace')
DEEPJOIN_DIR = SOURCE_DIR / 'output_deepjoin'

print(f"source dir   : {SOURCE_DIR}")
print(f"deepjoin dir : {DEEPJOIN_DIR}")

# ────────────────────────────────────────────────────────────────────
# 📍 라벨 표시명 매핑 (모든 차트/표에 자동 적용)
# ────────────────────────────────────────────────────────────────────
# 내부 라벨 (LABEL_PRESETS, JSON 키 등에 사용) → 표시 라벨 (차트/표 출력)
# 여기만 수정하면 모든 시각화에 일괄 적용됨.
LABEL_DISPLAY_NAMES = {
    # CRIM ablation 7 variants
    'CRIM-ablation | stat+char+ov': '제안모델',
    'CRIM-ablation | stat+char':    'stat+char',
    'CRIM-ablation | stat+ov':      'stat+ov',
    'CRIM-ablation | char+ov':      'char+ov',
    'CRIM-ablation | stat':         'stat',
    'CRIM-ablation | char':         'char',
    'CRIM-ablation | ov':           'ov',

    # 단일 모델
    'CRIM':                         '제안모델',
    'DeepJoin':                     'DeepJoin',
    'EmbDI':                        'EmbDI',
    'GPT-4o':                       'GPT',
    'GPT-4o | GPT-4o':              'GPT',

    # Valentine 3 matchers
    'Valentine-COMA | COMA':        'COMA',
    'Valentine-CUPID | CUPID':      'Cupid',
    'Valentine-DIST | DIST':        'DistributionBased',
}

def display_label(internal_label):
    """내부 라벨을 표시명으로 변환. 매핑 없으면 원본 그대로 반환."""
    return LABEL_DISPLAY_NAMES.get(internal_label, internal_label)


# ────────────────────────────────────────────────────────────────────
# 📍 라벨 색상 매핑 (모든 차트에 일관 적용)
# ────────────────────────────────────────────────────────────────────
# 같은 모델은 차트 종류 무관하게 같은 색상으로 표시되도록 중앙 관리.
LABEL_COLORS = {
    # CRIM ablation — Full 은 검정, 2-branch 는 중간 녹색, single 은 옅은 녹색
    'CRIM-ablation | stat+char+ov': '#222222',   # Full → 검정 (강조)
    'CRIM-ablation | stat+char':    '#2E7D32',
    'CRIM-ablation | stat+ov':      '#388E3C',
    'CRIM-ablation | char+ov':      '#43A047',
    'CRIM-ablation | stat':         '#81C784',
    'CRIM-ablation | char':         '#A5D6A7',
    'CRIM-ablation | ov':           '#C8E6C9',

    # 단일 모델
    'CRIM':                         '#8B0000',   # dark red
    'DeepJoin':                     '#1f77b4',   # blue
    'EmbDI':                        '#9467bd',   # purple
    'GPT-4o':                       '#FF6B6B',   # coral red

    # Valentine — 오렌지 계열
    'Valentine-COMA | COMA':        '#D2691E',
    'Valentine-CUPID | CUPID':      '#FF8C00',
    'Valentine-DIST | DIST':        '#FFA500',
}

# 표시명 → 내부 라벨 역매핑 (display 이름으로 lookup 가능하게)
_DISPLAY_TO_INTERNAL = {v: k for k, v in LABEL_DISPLAY_NAMES.items()}

def label_color(label, default='#888888'):
    """라벨에 대응하는 색상 반환. 내부/표시명 양쪽 지원."""
    if label in LABEL_COLORS:
        return LABEL_COLORS[label]
    internal = _DISPLAY_TO_INTERNAL.get(label)
    if internal and internal in LABEL_COLORS:
        return LABEL_COLORS[internal]
    return default


# ────────────────────────────────────────────────────────────────────
# 📍 모델 표시 순서 (모든 표/차트에 일괄 적용)
#    제안모델 → Cupid → COMA → DistributionBased → EmbDI → DeepJoin → GPT
# ────────────────────────────────────────────────────────────────────
MODEL_DISPLAY_ORDER = ['제안모델', 'Cupid', 'COMA', 'DistributionBased',
                       'EmbDI', 'DeepJoin', 'GPT']

def _model_order_rank(label):
    """내부 라벨/표시명 양쪽 지원 — 표시명 기준 순위 반환 (미정의는 맨 뒤)."""
    disp = LABEL_DISPLAY_NAMES.get(label, label)
    return MODEL_DISPLAY_ORDER.index(disp) if disp in MODEL_DISPLAY_ORDER \
        else len(MODEL_DISPLAY_ORDER)

def order_labels(labels, reverse=False):
    """라벨 리스트를 모델 표시 순서대로 정렬. reverse=True 면 역순(barh 용)."""
    out = sorted(labels, key=_model_order_rank)
    return out[::-1] if reverse else out

def reorder_models(frame, reverse=False):
    """DataFrame/Series 의 행(index)을 모델 표시 순서대로 재배열."""
    return frame.reindex(order_labels(list(frame.index), reverse=reverse))



In [ ]:
# === Per-format parsers ===
# 각 모델 JSON의 스키마가 달라 통일된 long-form DataFrame 로 변환

def parse_crim_single(path):
    """source/crim_ranking_metrics_{db}.json"""
    with open(path) as f:
        d = json.load(f)
    return [{
        'model':        'CRIM',
        'config':       d.get('model', 'CRIM_v34_final'),
        'db':           d['db'],
        'pr_auc':       d.get('pr_auc'),
        'recall_at_K':  d.get('recall_at_K'),
        'recall_at_3K': d.get('recall_at_3K'),
        'n_gt':         d.get('n_gt'),
        'total_pairs':  d.get('total_pairs'),
        'source_file':  os.path.basename(path),
    }]


def parse_crim_ablation(path):
    """source/ablation_ranking_metrics_*.json"""
    with open(path) as f:
        d = json.load(f)
    rows = []
    for tag, db_results in d.get('ranking_table', {}).items():
        for db, m in db_results.items():
            rows.append({
                'model':        'CRIM-ablation',
                'config':       tag,
                'db':           db,
                'pr_auc':       m.get('pr_auc'),
                'recall_at_K':  m.get('recall_at_K'),
                'recall_at_3K': m.get('recall_at_3K'),
                'n_gt':         m.get('n_gt'),
                'total_pairs':  m.get('total_pairs'),
                'source_file':  os.path.basename(path),
            })
    return rows


def parse_valentine(path):
    """source/valentine_ranking_metrics_{db}.json"""
    with open(path) as f:
        d = json.load(f)
    rows = []
    for matcher, m in d.get('per_model', {}).items():
        rows.append({
            'model':        f'Valentine-{matcher}',
            'config':       matcher,
            'db':           d.get('db'),
            'pr_auc':       m.get('pr_auc'),
            'recall_at_K':  m.get('recall_at_K'),
            'recall_at_3K': m.get('recall_at_3K'),
            'n_gt':         d.get('n_gt'),
            'total_pairs':  d.get('total_pairs'),
            'source_file':  os.path.basename(path),
        })
    return rows


def parse_deepjoin(path):
    """source/output_deepjoin/deepjoin_ranking_metrics.json"""
    with open(path) as f:
        d = json.load(f)
    rows = []
    for db, m in d.get('per_db', {}).items():
        rows.append({
            'model':        'DeepJoin',
            'config':       d.get('model', 'DeepJoin_VLDB2023'),
            'db':           db,
            'pr_auc':       m.get('pr_auc'),
            'recall_at_K':  m.get('recall_at_K'),
            'recall_at_3K': m.get('recall_at_3K'),
            'n_gt':         m.get('n_gt'),
            'total_pairs':  m.get('total_pairs'),
            'source_file':  os.path.basename(path),
        })
    return rows


def parse_embdi(path):
    """source/embdi_ranking_metrics_{db}.json"""
    with open(path) as f:
        d = json.load(f)
    return [{
        'model':        'EmbDI',
        'config':       d.get('model', 'EmbDI_Word2Vec'),
        'db':           d['db'],
        'pr_auc':       d.get('pr_auc'),
        'recall_at_K':  d.get('recall_at_K'),
        'recall_at_3K': d.get('recall_at_3K'),
        'n_gt':         d.get('n_gt'),
        'total_pairs':  d.get('total_pairs'),
        'source_file':  os.path.basename(path),
    }]


def parse_gpt4o(path):
    """source/gpt4o_ranking_metrics_{db}.json"""
    with open(path) as f:
        d = json.load(f)
    return [{
        'model':        'GPT-4o',
        'config':       d.get('model', 'GPT-4o'),
        'db':           d['db'],
        'pr_auc':       d.get('pr_auc'),
        'recall_at_K':  d.get('recall_at_K'),
        'recall_at_3K': d.get('recall_at_3K'),
        'n_gt':         d.get('n_gt'),
        'total_pairs':  d.get('total_pairs'),
        'source_file':  os.path.basename(path),
    }]


# === 자동 탐지 + 파싱 ===
all_rows = []
# ─── CRIM 버전 필터 ───────────────────────────────────────────────
# 파일명에 이 문자열이 포함된 CRIM/CRIM-ablation JSON 만 선택.
# 예: 'v35_22' → ablation_ranking_metrics_0524_v35_22_ablation.json 만 매칭.
# 빈 문자열 '' 이면 모든 버전 매칭 (필터 비활성).
# CRIM_VERSION_FILTER = 'v35_22'
CRIM_VERSION_FILTER = 'v34'
CRIM_VERSION_FILTER = 'v34_final'

discovered = defaultdict(list)

for path in [p for p in glob.glob(str(SOURCE_DIR / 'crim_ranking_metrics_*.json'))
             if not p.endswith('_classic.json') and CRIM_VERSION_FILTER in p]:
    discovered['CRIM'].append(path)
    all_rows.extend(parse_crim_single(path))

for path in [p for p in glob.glob(str(SOURCE_DIR / 'ablation_ranking_metrics_*.json'))
             if not p.endswith('_classic.json') and CRIM_VERSION_FILTER in p]:
    discovered['CRIM-ablation'].append(path)
    all_rows.extend(parse_crim_ablation(path))

for path in [p for p in glob.glob(str(SOURCE_DIR / 'valentine_ranking_metrics_*.json')) if not p.endswith('_classic.json')]:
    discovered['Valentine'].append(path)
    all_rows.extend(parse_valentine(path))

for path in [p for p in glob.glob(str(SOURCE_DIR / 'embdi_ranking_metrics_*.json')) if not p.endswith('_classic.json')]:
    discovered['EmbDI'].append(path)
    all_rows.extend(parse_embdi(path))

for path in [p for p in glob.glob(str(SOURCE_DIR / 'gpt4o_ranking_metrics_*.json')) if not p.endswith('_classic.json')]:
    discovered['GPT-4o'].append(path)
    all_rows.extend(parse_gpt4o(path))

if (DEEPJOIN_DIR / 'deepjoin_ranking_metrics.json').exists():
    p = str(DEEPJOIN_DIR / 'deepjoin_ranking_metrics.json')
    discovered['DeepJoin'].append(p)
    all_rows.extend(parse_deepjoin(p))

print("=== 탐지된 결과 파일 ===")
for model, paths in discovered.items():
    print(f"  [{model}] {len(paths)}개")
    for p in paths:
        print(f"    - {os.path.basename(p)}")

if not all_rows:
    print("\n⚠ 결과 파일이 없습니다. 먼저 각 노트북의 ranking 셀을 실행해주세요.")

In [ ]:
# === Long-form DataFrame ===
df = pd.DataFrame(all_rows)

# DB 이름 정규화: 마스킹 접미사(_mask)를 별도 컬럼으로 분리하여 비교 용이
df['db_base']    = df['db'].astype(str).str.replace(r'_mask$', '', regex=True)
df['is_masked']  = df['db'].astype(str).str.endswith('_mask')

# ────────────────────────────────────────────────────────────────────
# 📍 DB 필터 프리셋 (다운스트림 모든 셀에 자동 적용)
# ────────────────────────────────────────────────────────────────────
# 사용법: 아래 TARGET_DBS_PRESET 만 바꾸면 모든 표/차트/저장이 새 필터로 갱신됨.
#         원하는 프리셋이 없으면 DB_PRESETS 에 직접 추가 가능.

DB_PRESETS = {
    # 4개 원본 DB
    'original':         ['toxicology', 'california_schools', 'superhero', 'thrombosis_prediction'],
    # 4개 마스크 DB
    'masked':           ['toxicology_mask', 'california_schools_mask',
                         'superhero_mask',  'thrombosis_prediction_mask'],
    # 원본 + 마스크 8개 모두
    'all_8':            ['toxicology', 'california_schools', 'superhero', 'thrombosis_prediction',
                         'toxicology_mask', 'california_schools_mask',
                         'superhero_mask',  'thrombosis_prediction_mask'],
    # DB별 페어 비교 (원본 vs 마스크 나란히)
    'superhero_pair':   ['superhero',             'superhero_mask'],
    'toxicology_pair':  ['toxicology',            'toxicology_mask'],
    'california_pair':  ['california_schools',    'california_schools_mask'],
    'thrombosis_pair':  ['thrombosis_prediction', 'thrombosis_prediction_mask'],
    # 단일 DB 비교
    'only_superhero':   ['superhero'],
    'only_toxicology':  ['toxicology'],
}

# ★ 여기만 바꾸면 됨 ★
TARGET_DBS_PRESET = 'original'   # 'original' | 'masked' | 'all_8' | 'superhero_pair' | ...

TARGET_DBS = DB_PRESETS[TARGET_DBS_PRESET]
df = df[df['db'].isin(TARGET_DBS)].copy()

# label 컬럼 — 다운스트림 모든 셀(pivot, macro 등)에서 일관되게 사용
df['label'] = df.apply(
    lambda r: r['model'] if r['model'] in ('CRIM', 'DeepJoin', 'EmbDI', 'GPT-4o')
              else f"{r['model']} | {r['config']}",
    axis=1
)

# ────────────────────────────────────────────────────────────────────
# 📍 모델/config 필터 프리셋 (DB 필터와 동일한 패턴)
# ────────────────────────────────────────────────────────────────────
# label 단위로 필터 — 'CRIM', 'DeepJoin', 'CRIM-ablation | stat+char+ov' 등
# None 이면 전체 모델 표시

LABEL_PRESETS = {
    'all':                None,   # 전체

    # 메인 비교 — 주요 baseline 6개 (ablation 제외)
    'main_models':        ['CRIM', 'CRIM-ablation | stat+char+ov',  # 둘 중 데이터에 있는 쪽 자동 매칭
                           'DeepJoin', 'EmbDI',
                           'Valentine-COMA | COMA',
                           'Valentine-CUPID | CUPID',
                           'Valentine-DIST | DIST',
                           'GPT-4o'],

    # CRIM Full + 다른 모델들 (ablation 의 stat+char+ov 를 CRIM 대체로 사용)
    'crim_full_vs_all':   ['CRIM-ablation | stat+char+ov', 'DeepJoin', 'EmbDI',
                           'Valentine-COMA | COMA',
                           'Valentine-CUPID | CUPID',
                           'Valentine-DIST | DIST'],

    # Ablation 만 (7개 variant)
    'ablation_only':      ['CRIM-ablation | stat',
                           'CRIM-ablation | char',
                           'CRIM-ablation | ov',
                           'CRIM-ablation | stat+char',
                           'CRIM-ablation | stat+ov',
                           'CRIM-ablation | char+ov',
                           'CRIM-ablation | stat+char+ov'],

    # CRIM 계열만 (main + ablation)
    'crim_only':          ['CRIM',
                           'CRIM-ablation | stat',
                           'CRIM-ablation | char',
                           'CRIM-ablation | ov',
                           'CRIM-ablation | stat+char',
                           'CRIM-ablation | stat+ov',
                           'CRIM-ablation | char+ov',
                           'CRIM-ablation | stat+char+ov'],

    # 최고 성능 2개만 (PR-AUC top)
    'top_2':              ['CRIM-ablation | stat+char+ov', 'DeepJoin'],

    # Valentine 3개 + CRIM Full
    'valentine_vs_crim':  ['CRIM-ablation | stat+char+ov',
                           'Valentine-COMA | COMA',
                           'Valentine-CUPID | CUPID',
                           'Valentine-DIST | DIST'],
}

# ★ 사용할 프리셋 (여기만 바꾸면 됨) ★
LABEL_PRESET = 'main_models'   # 'all' | 'main_models' | 'crim_full_vs_all' | 'ablation_only'
                       # | 'crim_only' | 'top_2' | 'valentine_vs_crim'

# 필터 적용
if LABEL_PRESETS[LABEL_PRESET] is not None:
    _before = len(df)
    df = df[df['label'].isin(LABEL_PRESETS[LABEL_PRESET])].copy()
    print(f"📍 LABEL 필터 프리셋 : '{LABEL_PRESET}'")
    print(f"   적용된 라벨       : {LABEL_PRESETS[LABEL_PRESET]}")
    print(f"   필터 후 행 수     : {_before} → {len(df)}")
else:
    print(f"📍 LABEL 필터 프리셋 : '{LABEL_PRESET}' (필터 없음 — 전체 모델)")

# ── 라벨을 표시명으로 변환 (LABEL_PRESETS 필터 이후) ──
# 내부 'Valentine-COMA | COMA' → 표시 'COMA' 등
if 'LABEL_DISPLAY_NAMES' in dir():
    df['label'] = df['label'].map(lambda x: LABEL_DISPLAY_NAMES.get(x, x))

# 핵심 컬럼 정렬
cols = ['model', 'config', 'db', 'db_base', 'is_masked', 'label',
        'pr_auc', 'recall_at_K', 'recall_at_3K', 'n_gt', 'total_pairs', 'source_file']
df = df[cols].sort_values(['db_base', 'model', 'config']).reset_index(drop=True)

# ── 중복 제거 (같은 (model, config, db) 가 여러 JSON 파일에 있을 경우 대비) ──
n_before = len(df)
df = df.drop_duplicates(subset=['model', 'config', 'db'], keep='first').reset_index(drop=True)
if n_before != len(df):
    print(f"⚠  중복 제거: {n_before} → {len(df)} ({n_before - len(df)}개 제거)")

print(f"📍 DB 필터 프리셋  : '{TARGET_DBS_PRESET}'")
print(f"   적용된 DB       : {TARGET_DBS}")
print(f"   필터 후 행 수   : {len(df)}")
print(f"   고유 모델 수    : {df['model'].nunique()}")
print(f"   고유 DB 수      : {df['db'].nunique()}")
print(f"=== 필터 후 모델별 DB 개수 ===")
print(df.groupby('model')['db'].nunique().to_string())

# ── (옵션 B 비활성화) — 원본 라벨 그대로 유지 ─────────────────────
# CRIM-ablation | stat+char+ov 행은 원래 라벨대로 표에 등장
# 메인 CRIM 행도 그대로 유지 (사용자가 원할 경우 비교 가능)
df


In [ ]:
# === Wide table: PR-AUC (row=model/config, col=db) ===
print("=" * 80)
print("[PR-AUC: model × db]")
print("=" * 80)

# CRIM-ablation 의 config 가 많아 row label 이 길 수 있음 → 'model | config' 결합
df['label'] = df.apply(
    lambda r: r['model'] if r['model'] in ('CRIM', 'DeepJoin', 'EmbDI')
              else f"{r['model']} | {r['config']}",
    axis=1
)

pivot_prauc = df.pivot_table(index='label', columns='db', values='pr_auc',
                             aggfunc='first')
pivot_prauc['mean']  = pivot_prauc.mean(axis=1, skipna=True)
pivot_prauc = reorder_models(pivot_prauc)   # 모델 표시 순서로 정렬
# NaN 셀은 '-' 로 표시
display(pivot_prauc.style.format('{:.4f}', na_rep='N/A'))

In [ ]:
# === Wide table: Recall@K ===
print("=" * 80)
print("[Recall@K (K=|GT|): model × db]")
print("=" * 80)

pivot_rk = df.pivot_table(index='label', columns='db', values='recall_at_K',
                         aggfunc='first')
pivot_rk['mean']  = pivot_rk.mean(axis=1, skipna=True)
pivot_rk = reorder_models(pivot_rk)   # 모델 표시 순서로 정렬
display(pivot_rk.style.format('{:.4f}', na_rep='N/A'))

In [ ]:
# === 모델별 매크로 평균 + 순위 ===
print("=" * 80)
print("[모델별 macro 평균 — DB 평균]")
print("=" * 80)

macro = df.groupby('label').agg(
    avg_pr_auc       = ('pr_auc',       'mean'),
    avg_recall_at_K  = ('recall_at_K',  'mean'),
    avg_recall_at_3K = ('recall_at_3K', 'mean'),
    n_dbs            = ('db',           'nunique'),
).round(4)
macro = reorder_models(macro)   # 모델 표시 순서로 정렬
display(macro.style.format('{:.4f}', na_rep='N/A'))

print("\n[PR-AUC 기준 Top-10]")
print(macro.sort_values('avg_pr_auc', ascending=False).head(10).to_string())

print("\n[Recall@K 기준 Top-10]")
print(macro.sort_values('avg_recall_at_K', ascending=False).head(10).to_string())

In [ ]:
# === 종합 표: Method × DB × (PR-AUC, R@K, R@3K) 한 화면 ─────────────
# Method 가 row, DB 가 outer column, 3개 지표가 inner column 인 wide 표
# 각 (DB, metric) 컬럼에서 최댓값을 굵게 + 색 강조

if df.empty:
    print("(no ranking data)")
else:
    # 1) Pivot: index=label, columns=(metric, db) → swap → (db, metric)
    wide = df.pivot_table(
        index='label',
        columns='db',
        values=['pr_auc', 'recall_at_K', 'recall_at_3K'],
        aggfunc='first'
    )
    # 컬럼 레벨 순서를 (db, metric) 으로 변경
    wide.columns = wide.columns.swaplevel(0, 1)
    wide = wide.sort_index(axis=1, level=0)

    # 2) 메트릭 라벨 단축
    metric_rename = {'pr_auc': 'PR-AUC',
                     'recall_at_K': 'R@K',
                     'recall_at_3K': 'R@3K'}
    wide.columns = pd.MultiIndex.from_tuples(
        [(db, metric_rename.get(m, m)) for db, m in wide.columns]
    )

    # 3) DB별 평균을 마지막 그룹으로 추가
    avg_block = pd.DataFrame({
        ('Avg', 'PR-AUC'):  wide.xs('PR-AUC', axis=1, level=1).mean(axis=1, skipna=True),
        ('Avg', 'R@K'):     wide.xs('R@K',    axis=1, level=1).mean(axis=1, skipna=True),
        ('Avg', 'R@3K'):    wide.xs('R@3K',   axis=1, level=1).mean(axis=1, skipna=True),
    })
    avg_block.columns = pd.MultiIndex.from_tuples(avg_block.columns)
    wide_full = pd.concat([wide, avg_block], axis=1)

    # 4) Avg PR-AUC 기준 정렬 (CRIM 같은 강한 모델이 위로)
    wide_full = reorder_models(wide_full)   # 모델 표시 순서로 정렬

    # 5) Styling: 컬럼별 max bold + 컬러 그라디언트
    def _bold_max(col):
        is_max = col == col.max()
        return ['font-weight: bold' if v else '' for v in is_max]

    styled = (wide_full.style
              .format("{:.3f}", na_rep="N/A")
              .apply(_bold_max, axis=0)
              .background_gradient(cmap='YlGn', axis=0, low=0.1, high=0.9)
              .set_caption("All ranking metrics — Method × Database × {PR-AUC, R@K, R@3K}. "
                           "Bold = column-wise best. Sorted by Avg PR-AUC.")
              .set_table_styles([
                  {'selector': 'th.col_heading.level0',
                   'props': 'text-align: center; border-bottom: 2px solid #888;'},
                  {'selector': 'th.col_heading.level1',
                   'props': 'text-align: center; font-size: 0.9em; color: #555;'},
                  {'selector': 'caption',
                   'props': 'caption-side: top; font-weight: bold; '
                            'text-align: left; padding: 6px;'},
              ]))
    display(styled)

    # ─── (추가) 최종 macro 요약 표 — 논문/슬라이드 직접 사용용 ───────────
    macro_summary = pd.DataFrame({
        'PR-AUC': wide.xs('PR-AUC', axis=1, level=1).mean(axis=1, skipna=True),
        'R@K':    wide.xs('R@K',    axis=1, level=1).mean(axis=1, skipna=True),
        'R@3K':   wide.xs('R@3K',   axis=1, level=1).mean(axis=1, skipna=True),
    })
    macro_summary = reorder_models(macro_summary)   # 모델 표시 순서로 정렬

    # CRIM 행 라벨에 (ours) 표기 (제안 모델 강조)
    macro_summary.index = [
        f"{m} (ours)" if m == 'CRIM' else m
        for m in macro_summary.index
    ]
    macro_summary.index.name = 'Model'

    styled_macro = (macro_summary.style
                    .format("{:.4f}", na_rep="N/A")
                    .apply(_bold_max, axis=0)
                    .background_gradient(cmap='YlGn', axis=0, low=0.1, high=0.9)
                    .set_caption("최종 요약 — 모델별 macro 평균 (DB 평균). "
                                 "Bold = 컬럼별 최댓값. 정렬: PR-AUC 내림차순.")
                    .set_table_styles([
                        {'selector': 'caption',
                         'props': 'caption-side: top; font-weight: bold; '
                                  'text-align: left; padding: 6px;'},
                        {'selector': 'th',
                         'props': 'text-align: center; padding: 6px 12px;'},
                        {'selector': 'td',
                         'props': 'text-align: center; padding: 4px 12px;'},
                    ]))
    print()  # 위 표와 시각적 분리
    display(styled_macro)

    # plain DataFrame 도 함께 출력 (복사·붙여넣기 / CSV 변환용)
    print()
    print("=" * 60)
    print("[Plain text — 복사용]")
    print("=" * 60)
    print(macro_summary.round(4).fillna("N/A").to_string())


In [ ]:
# === 시각화: PR-AUC 막대 차트 (model × db) ===
if not df.empty:
    # 두 차트에서 색상 일치를 위해 레이블 순서와 색상을 고정
    label_order = pivot_prauc.drop(columns=['mean'], errors='ignore').index.tolist()
    palette = plt.cm.tab10.colors
    color_map = {label: palette[i % len(palette)] for i, label in enumerate(label_order)}

    plot_df = pivot_prauc.drop(columns=['mean'], errors='ignore').reindex(label_order)
    if not plot_df.empty:
        n_models, n_dbs = plot_df.shape
        fig, ax = plt.subplots(figsize=(max(8, 1.0*n_dbs*n_models/4), 6))
        colors = [color_map[label] for label in plot_df.index]
        plot_df.T.plot(kind='bar', ax=ax, width=0.85, color=colors)
        ax.set_title('PR-AUC per DB across models', fontsize=13, fontweight='bold')
        ax.set_ylabel('PR-AUC')
        ax.set_xlabel('DB')
        ax.set_ylim(0, 1.0)
        ax.grid(axis='y', alpha=0.3)
        ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0), fontsize=8)
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("(no data to plot)")

In [ ]:
# === 시각화: Recall@K 막대 차트 ===
if not df.empty:
    # cell-7(PR-AUC)에서 정의한 label_order, color_map 재사용 → 색상 일치
    plot_df = pivot_rk.drop(columns=['mean'], errors='ignore').reindex(label_order)
    if not plot_df.empty:
        n_models, n_dbs = plot_df.shape
        fig, ax = plt.subplots(figsize=(max(8, 1.0*n_dbs*n_models/4), 6))
        colors = [color_map[label] for label in plot_df.index]
        plot_df.T.plot(kind='bar', ax=ax, width=0.85, color=colors)
        ax.set_title('Recall@K (K=|GT|) per DB across models',
                     fontsize=13, fontweight='bold')
        ax.set_ylabel('Recall@K')
        ax.set_xlabel('DB')
        ax.set_ylim(0, 1.05)
        ax.grid(axis='y', alpha=0.3)
        ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0), fontsize=8)
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("(no data to plot)")

In [ ]:
# === 통합 결과 저장 (CSV + JSON) ===
out_dir = SOURCE_DIR / 'integrated_results'
out_dir.mkdir(exist_ok=True)

csv_path  = out_dir / 'all_models_long.csv'
df.to_csv(csv_path, index=False)
print(f"💾 long-form CSV : {csv_path}")

pivot_prauc.to_csv(out_dir / 'pivot_pr_auc.csv')
pivot_rk.to_csv(out_dir / 'pivot_recall_at_K.csv')
macro.to_csv(out_dir / 'macro_average.csv')
print(f"💾 wide / macro tables → {out_dir}/")

json_path = out_dir / 'all_models_summary.json'
with open(json_path, 'w') as f:
    json.dump({
        'macro_average': macro.reset_index().to_dict(orient='records'),
        'pr_auc_pivot':  pivot_prauc.reset_index().to_dict(orient='records'),
        'recall_at_K_pivot': pivot_rk.reset_index().to_dict(orient='records'),
        'long_records':  df.to_dict(orient='records'),
    }, f, indent=2, default=lambda x: None if pd.isna(x) else x)
print(f"💾 통합 JSON     : {json_path}")

## Classic Metrics 비교 (TP/FP/FN/TN, Precision/Recall/F1)

각 모델 노트북에서 생성된 `*_ranking_metrics_*_classic.json` 파일을 자동 탐지·통합하여
**임계값 기반** 분류 지표를 비교한다. 위의 PR-AUC / Recall@K 결과(임계값-독립)와
**옆에 두고 보면 같은 모델이 어느 임계값에서 어떤 동작을 하는지** 파악 가능.

각 모델별 임계값:
- CRIM, CRIM-ablation : 0.5 (sigmoid)
- Valentine : 0.5 (matcher score)
- DeepJoin : 0.70 (cosine similarity, `SIMILARITY_THRESHOLD`)
- EmbDI : 0.98 (Word2Vec cosine, `run_evaluation` default)

→ 임계값이 모델마다 다르다는 점을 비교 표 해석 시 항상 명시할 것.

In [ ]:
# === Classic JSON parsers ===
# 5종 _classic.json 파일을 통일된 long-form 으로 변환

def parse_crim_single_classic(path):
    with open(path) as f:
        d = json.load(f)
    return [{
        'model':       'CRIM',
        'config':      d.get('model', 'CRIM_v34_final'),
        'db':          d['db'],
        'threshold':   d.get('threshold'),
        'tp':          d.get('tp'), 'fp': d.get('fp'),
        'fn':          d.get('fn'), 'tn': d.get('tn'),
        'precision':   d.get('precision'),
        'recall':      d.get('recall'),
        'f1':          d.get('f1'),
        'n_gt':        d.get('n_gt'),
        'total_pairs': d.get('total_pairs'),
        'source_file': os.path.basename(path),
    }]


def parse_crim_ablation_classic(path):
    with open(path) as f:
        d = json.load(f)
    rows = []
    thr = d.get('threshold')
    for tag, db_results in d.get('results_table', {}).items():
        for db, m in db_results.items():
            rows.append({
                'model':       'CRIM-ablation',
                'config':      tag,
                'db':          db,
                'threshold':   thr,
                'tp':          m.get('tp'), 'fp': m.get('fp'),
                'fn':          m.get('fn'), 'tn': m.get('tn'),
                'precision':   m.get('precision'),
                'recall':      m.get('recall'),
                'f1':          m.get('f1'),
                'n_gt':        None,
                'total_pairs': None,
                'source_file': os.path.basename(path),
            })
    return rows


def parse_valentine_classic(path):
    with open(path) as f:
        d = json.load(f)
    rows = []
    thr = d.get('threshold')
    db  = d.get('db')
    for matcher, m in d.get('per_model', {}).items():
        rows.append({
            'model':       f'Valentine-{matcher}',
            'config':      matcher,
            'db':          db,
            'threshold':   thr,
            'tp':          m.get('tp'), 'fp': m.get('fp'),
            'fn':          m.get('fn'), 'tn': m.get('tn'),
            'precision':   m.get('precision'),
            'recall':      m.get('recall'),
            'f1':          m.get('f1'),
            'n_gt':        d.get('n_gt'),
            'total_pairs': m.get('total_pairs'),
            'source_file': os.path.basename(path),
        })
    return rows


def parse_deepjoin_classic(path):
    with open(path) as f:
        d = json.load(f)
    rows = []
    thr = d.get('threshold')
    for db, m in d.get('per_db', {}).items():
        rows.append({
            'model':       'DeepJoin',
            'config':      d.get('model', 'DeepJoin_VLDB2023'),
            'db':          db,
            'threshold':   thr,
            'tp':          m.get('tp'), 'fp': m.get('fp'),
            'fn':          m.get('fn'), 'tn': m.get('tn'),
            'precision':   m.get('precision'),
            'recall':      m.get('recall'),
            'f1':          m.get('f1'),
            'n_gt':        None,
            'total_pairs': None,
            'source_file': os.path.basename(path),
        })
    return rows


def parse_embdi_classic(path):
    with open(path) as f:
        d = json.load(f)
    return [{
        'model':       'EmbDI',
        'config':      d.get('model', 'EmbDI_Word2Vec'),
        'db':          d['db'],
        'threshold':   d.get('threshold'),
        'tp':          d.get('tp'), 'fp': d.get('fp'),
        'fn':          d.get('fn'), 'tn': d.get('tn'),
        'precision':   d.get('precision'),
        'recall':      d.get('recall'),
        'f1':          d.get('f1'),
        'n_gt':        d.get('n_gt'),
        'total_pairs': d.get('total_pairs'),
        'source_file': os.path.basename(path),
    }]


# === 자동 탐지 + 파싱 ===
all_rows_classic = []
discovered_classic = defaultdict(list)

# CRIM_VERSION_FILTER 가 정의돼 있으면 적용 (cell 2 와 동일 로직)
_crim_ver_cls = globals().get('CRIM_VERSION_FILTER', '')

for path in glob.glob(str(SOURCE_DIR / 'crim_ranking_metrics_*_classic.json')):
    if _crim_ver_cls and _crim_ver_cls not in path:
        continue
    discovered_classic['CRIM'].append(path)
    all_rows_classic.extend(parse_crim_single_classic(path))

for path in glob.glob(str(SOURCE_DIR / 'ablation_ranking_metrics_*_classic.json')):
    if _crim_ver_cls and _crim_ver_cls not in path:
        continue
    discovered_classic['CRIM-ablation'].append(path)
    all_rows_classic.extend(parse_crim_ablation_classic(path))

for path in glob.glob(str(SOURCE_DIR / 'valentine_ranking_metrics_*_classic.json')):
    discovered_classic['Valentine'].append(path)
    all_rows_classic.extend(parse_valentine_classic(path))

for path in glob.glob(str(SOURCE_DIR / 'embdi_ranking_metrics_*_classic.json')):
    discovered_classic['EmbDI'].append(path)
    all_rows_classic.extend(parse_embdi_classic(path))

if (DEEPJOIN_DIR / 'deepjoin_ranking_metrics_classic.json').exists():
    p = str(DEEPJOIN_DIR / 'deepjoin_ranking_metrics_classic.json')
    discovered_classic['DeepJoin'].append(p)
    all_rows_classic.extend(parse_deepjoin_classic(p))

print("=== 탐지된 classic 결과 파일 ===")
for model, paths in discovered_classic.items():
    print(f"  [{model}] {len(paths)}개")
    for p in paths:
        print(f"    - {os.path.basename(p)}")

if not all_rows_classic:
    print("\n⚠ classic 결과 파일이 없습니다. 먼저 각 노트북의 classic 셀을 실행해주세요.")

In [ ]:
# === Long-form DataFrame (classic) ===
df_classic = pd.DataFrame(all_rows_classic)

if not df_classic.empty:
    df_classic['db_base']   = df_classic['db'].astype(str).str.replace(r'_mask$', '', regex=True)
    df_classic['is_masked'] = df_classic['db'].astype(str).str.endswith('_mask')

    # ── cell 3 에서 정의한 TARGET_DBS 재사용 (단일 소스 보장) ───────
    # cell 3 미실행 환경에서도 동작하도록 폴백 추가
    if 'TARGET_DBS' not in dir():
        # fallback — cell 3 의 'masked' 프리셋과 동일
        TARGET_DBS = ['toxicology_mask', 'california_schools_mask',
                      'superhero_mask',  'thrombosis_prediction_mask']
        print("⚠ TARGET_DBS 미정의 — fallback (masked) 사용. cell 3 부터 실행 권장")

    df_classic = df_classic[df_classic['db'].isin(TARGET_DBS)].copy()

    df_classic['label'] = df_classic.apply(
        lambda r: r['model'] if r['model'] in ('CRIM', 'DeepJoin', 'EmbDI')
                  else f"{r['model']} | {r['config']}",
        axis=1
    )

    # ── cell 3 의 LABEL_PRESET 재사용 (단일 소스) ─────────────
    if 'LABEL_PRESETS' in dir() and 'LABEL_PRESET' in dir():
        if LABEL_PRESETS[LABEL_PRESET] is not None:
            _b = len(df_classic)
            df_classic = df_classic[df_classic['label'].isin(LABEL_PRESETS[LABEL_PRESET])].copy()
            print(f"📍 (classic) LABEL 필터 '{LABEL_PRESET}': {_b} → {len(df_classic)}")

    # 라벨을 표시명으로 변환 (필터 이후)
    if 'LABEL_DISPLAY_NAMES' in dir():
        df_classic['label'] = df_classic['label'].map(lambda x: LABEL_DISPLAY_NAMES.get(x, x))
    cols = ['model', 'config', 'db', 'db_base', 'is_masked', 'threshold',
            'tp', 'fp', 'fn', 'tn', 'precision', 'recall', 'f1',
            'n_gt', 'total_pairs', 'label', 'source_file']
    df_classic = df_classic[cols].sort_values(['db_base', 'model', 'config']).reset_index(drop=True)

    # ── 중복 제거 ──────────────────────────────────────────────
    n_before = len(df_classic)
    df_classic = df_classic.drop_duplicates(subset=['model', 'config', 'db'], keep='first').reset_index(drop=True)
    if n_before != len(df_classic):
        print(f"⚠  중복 제거: {n_before} → {len(df_classic)} ({n_before - len(df_classic)}개 제거)")

print(f"📍 DB 필터 적용  : {TARGET_DBS}")
print(f"   필터 후 행 수 : {len(df_classic)}")

# ── (옵션 B 비활성화) — classic 측도 원본 라벨 유지 ──────────────
df_classic


In [ ]:
# === Wide table: F1 (row=model/config, col=db) ===
print("=" * 80)
print("[F1 @ threshold: model × db]")
print("=" * 80)

if not df_classic.empty:
    pivot_f1 = df_classic.pivot_table(index='label', columns='db', values='f1', aggfunc='first')
    pivot_f1['mean'] = pivot_f1.mean(axis=1, skipna=True)
    pivot_f1 = reorder_models(pivot_f1)   # 모델 표시 순서로 정렬
    display(pivot_f1.style.format('{:.4f}', na_rep='N/A'))
else:
    print("(no classic data)")

In [ ]:
# === Wide table: Precision & Recall ===
if not df_classic.empty:
    print("=" * 80)
    print("[Precision @ threshold: model × db]")
    print("=" * 80)
    pivot_p = df_classic.pivot_table(index='label', columns='db', values='precision', aggfunc='first')
    pivot_p['mean'] = pivot_p.mean(axis=1, skipna=True)
    pivot_p = reorder_models(pivot_p)   # 모델 표시 순서로 정렬
    display(pivot_p.style.format('{:.4f}', na_rep='N/A'))

    print("\n" + "=" * 80)
    print("[Recall @ threshold: model × db]")
    print("=" * 80)
    pivot_r = df_classic.pivot_table(index='label', columns='db', values='recall', aggfunc='first')
    pivot_r['mean'] = pivot_r.mean(axis=1, skipna=True)
    pivot_r = reorder_models(pivot_r)   # 모델 표시 순서로 정렬
    display(pivot_r.style.format('{:.4f}', na_rep='N/A'))

In [ ]:
# === 모델별 macro 평균 (classic) ===
if not df_classic.empty:
    print("=" * 80)
    print("[모델별 macro 평균 — classic metrics]")
    print("=" * 80)
    macro_classic = df_classic.groupby('label').agg(
        avg_precision = ('precision', 'mean'),
        avg_recall    = ('recall',    'mean'),
        avg_f1        = ('f1',        'mean'),
        sum_tp        = ('tp',        'sum'),
        sum_fp        = ('fp',        'sum'),
        sum_fn        = ('fn',        'sum'),
        sum_tn        = ('tn',        'sum'),
        n_dbs         = ('db',        'nunique'),
    ).round(4)
    macro_classic = reorder_models(macro_classic)   # 모델 표시 순서로 정렬
    display(macro_classic.style.format('{:.4f}', na_rep='N/A'))

In [ ]:
# === 시각화: F1 막대 차트 ===
if not df_classic.empty:
    plot_df = pivot_f1.drop(columns=['mean'], errors='ignore')
    if not plot_df.empty:
        n_models, n_dbs = plot_df.shape
        # fig, ax = plt.subplots(figsize=(max(5, 1.0*n_dbs*n_models/3), 5))
        fig, ax = plt.subplots(figsize=(9, 5))
        plot_df.T.plot(kind='bar', ax=ax, width=0.55)
        # ax.set_title('F1 @ threshold per DB across models',
        #              fontsize=13, fontweight='bold')
        ax.set_ylabel('F1-score')
        ax.set_xlabel('Dataset')
        ax.set_ylim(0, 1.05)
        ax.grid(axis='y', alpha=0.3)
        ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0), fontsize=8)
        # plt.xticks(rotation=30, ha='right')
        plt.xticks(rotation=0, ha='center')
        plt.tight_layout()
        plt.show()
else:
    print("(no data to plot)")

In [ ]:
# === Ranking vs Classic side-by-side (label 별 PR-AUC vs F1) ===
# 두 지표가 같은 (model, config, db) 에 대해 어떻게 다른지 한 표에서 확인
if not df.empty and not df_classic.empty:
    rk_view = df[['label', 'db', 'pr_auc', 'recall_at_K']].rename(
        columns={'pr_auc': 'pr_auc', 'recall_at_K': 'recall_at_K'}
    )
    cl_view = df_classic[['label', 'db', 'precision', 'recall', 'f1', 'threshold']]
    combined = rk_view.merge(cl_view, on=['label', 'db'], how='outer')
    # label 을 모델 표시 순서로 categorical 정렬
    _lab_order = order_labels(list(pd.unique(combined['label'].dropna())))
    combined['label'] = pd.Categorical(combined['label'], categories=_lab_order, ordered=True)
    combined = combined.sort_values(['db', 'label']).reset_index(drop=True)

    print("=" * 100)
    print("[Ranking vs Classic — 같은 (model, db) 쌍의 지표 비교]")
    print("=" * 100)
    display(combined.style.format({'pr_auc': '{:.4f}', 'recall_at_K': '{:.4f}',
                              'precision': '{:.4f}', 'recall': '{:.4f}',
                              'f1': '{:.4f}', 'threshold': '{:.2f}'},
                              na_rep='N/A'))
else:
    print("(ranking 또는 classic 데이터 부족)")

In [ ]:
# === 통합 결과 저장 (classic) ===
out_dir = SOURCE_DIR / 'integrated_results'
out_dir.mkdir(exist_ok=True)

if not df_classic.empty:
    df_classic.to_csv(out_dir / 'all_models_classic_long.csv', index=False)
    pivot_f1.to_csv(out_dir / 'pivot_f1.csv')
    if 'pivot_p' in dir(): pivot_p.to_csv(out_dir / 'pivot_precision.csv')
    if 'pivot_r' in dir(): pivot_r.to_csv(out_dir / 'pivot_recall.csv')
    macro_classic.to_csv(out_dir / 'macro_classic.csv')

    # Combined ranking + classic in one JSON
    json_path = out_dir / 'all_models_classic_summary.json'
    with open(json_path, 'w') as f:
        json.dump({
            'macro_classic': macro_classic.reset_index().to_dict(orient='records'),
            'f1_pivot':      pivot_f1.reset_index().to_dict(orient='records'),
            'long_records':  df_classic.to_dict(orient='records'),
        }, f, indent=2, default=lambda x: None if pd.isna(x) else x)
    print(f"💾 classic long CSV  : {out_dir/'all_models_classic_long.csv'}")
    print(f"💾 pivot/macro CSV   : {out_dir}/")
    print(f"💾 classic JSON 통합  : {json_path}")

    # Combined ranking + classic single table
    if not df.empty:
        rk_subset = df[['label', 'db', 'pr_auc', 'recall_at_K', 'recall_at_3K']]
        cl_subset = df_classic[['label', 'db', 'precision', 'recall', 'f1', 'threshold']]
        merged = rk_subset.merge(cl_subset, on=['label', 'db'], how='outer')
        merged.to_csv(out_dir / 'all_models_ranking_plus_classic.csv', index=False)
        print(f"💾 통합 (ranking + classic): {out_dir/'all_models_ranking_plus_classic.csv'}")

## 사용 메모

- **새 결과가 추가되면**: 해당 모델 노트북의 ranking 셀을 다시 실행 → 같은 JSON 경로가 덮어쓰여짐 → 이 노트북의 모든 셀을 위에서부터 다시 실행.
- **DB 이름 일치**: 모든 모델이 같은 `db_name` 으로 평가해야 한 열에 모임. 마스킹된 DB는 `db_base` 컬럼으로 비교.
- **EmbDI 주의**: `skipped_keyerror`가 크면 `n_gt`가 줄어들어 다른 모델 대비 K가 작아질 수 있음 — 원본 JSON에서 확인.
- **DeepJoin 주의**: directional score를 frozenset max 로 대칭화함 — 비교 protocol은 일치하나 max 대신 mean 도 가능한 자유도가 있음 (논문 footnote에 명시 권장).

## 최종 시각화 — PR-AUC (표 + Heatmap) / Recall@K · Recall@3K (Bar chart)

각 지표를 두 가지 단위로 표현:
- **모델 × DB 별** (2D): 세부 분포 확인용
- **모델별** (1D, DB macro 평균): 순위 한눈 확인용

PR-AUC 는 색상 강조에 적합한 heatmap, Recall@K/R@3K 는 정량 비교가 중요한 bar chart 사용.

In [ ]:
def _render_heatmap(heat_data, metric_name, cmap='YlGn'):
    """PR-AUC 형식과 동일한 heatmap — model × DB"""
    import matplotlib.pyplot as plt
    import numpy as np
    try:
        import seaborn as sns
        HAS_SEABORN_LOCAL = True
    except ImportError:
        HAS_SEABORN_LOCAL = False

    if heat_data.empty:
        return
    n_models, n_dbs = heat_data.shape
    fig, ax = plt.subplots(figsize=(max(8, 1.2*n_dbs), max(5, 0.5*n_models)))
    if HAS_SEABORN_LOCAL:
        sns.heatmap(heat_data, annot=True, fmt='.3f', cmap=cmap,
                    vmin=0, vmax=1, cbar_kws={'label': metric_name},
                    annot_kws={'size': 15},
                    linewidths=0.5, linecolor='white', ax=ax)
        ax.tick_params(axis='both', labelsize=15)
        plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
    else:
        im = ax.imshow(heat_data.values, cmap=cmap, vmin=0, vmax=1, aspect='auto')
        ax.set_xticks(range(n_dbs))
        ax.set_xticklabels(heat_data.columns, rotation=20, ha='right', fontsize=15)
        ax.set_yticks(range(n_models))
        ax.set_yticklabels(heat_data.index, fontsize=15)
        for _i in range(n_models):
            for _j in range(n_dbs):
                v = heat_data.values[_i, _j]
                if not np.isnan(v):
                    ax.text(_j, _i, f'{v:.3f}', ha='center', va='center',
                            color='white' if v > 0.5 else 'black', fontsize=15)
        plt.colorbar(im, ax=ax, label=metric_name)
    ax.set_xlabel('Database', fontsize=15); ax.set_ylabel('Model', fontsize=15)
    ax.set_ylabel('')
    ax.tick_params(axis='y', labelleft=False)
    plt.tight_layout()
    plt.show()

# === PR-AUC: 표 + Heatmap (2가지 버전) ===
import matplotlib.pyplot as plt
import numpy as np

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

if df.empty:
    print("(no ranking data)")
else:
    # ── (1) 모델 × DB 표 ──────────────────────────────────────
    print("=" * 80)
    print("[PR-AUC] 모델 × DB 표")
    print("=" * 80)
    pivot_prauc_view = df.pivot_table(index='label', columns='db', values='pr_auc',
                                       aggfunc='first')
    pivot_prauc_view['mean'] = pivot_prauc_view.mean(axis=1, skipna=True)
    pivot_prauc_view = reorder_models(pivot_prauc_view)   # 모델 표시 순서로 정렬

    # ── 인덱스(라벨)를 표시명으로 변환 (idempotent) ──
    if 'LABEL_DISPLAY_NAMES' in dir():
        pivot_prauc_view.index = [LABEL_DISPLAY_NAMES.get(l, l)
                                  for l in pivot_prauc_view.index]

    display(pivot_prauc_view.style
            .format('{:.4f}', na_rep='N/A')
            .background_gradient(cmap='YlGn', axis=None, low=0.0, high=1.0)
            .set_caption('PR-AUC per model × DB (sorted by macro avg)')
            .set_table_styles([
                {'selector': 'caption',
                 'props': 'caption-side: top; font-weight: bold; padding: 6px;'},
            ]))

    # ── (2) 모델 × DB Heatmap ─────────────────────────────────
    heat_data = pivot_prauc_view.drop(columns=['mean'], errors='ignore')
    if not heat_data.empty:
        n_models, n_dbs = heat_data.shape
        fig, ax = plt.subplots(figsize=(max(8, 1.2*n_dbs), max(5, 0.5*n_models)))

        if HAS_SEABORN:
            sns.heatmap(heat_data, annot=True, fmt='.3f', cmap='YlGn',
                        vmin=0, vmax=1, cbar_kws={'label': 'PR-AUC'},
                        annot_kws={'size': 15},
                        linewidths=0.5, linecolor='white', ax=ax)
            # tick label 폰트 + x축 회전 (겹침 방지)
            ax.tick_params(axis='both', labelsize=15)
            plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
        else:
            # matplotlib fallback
            im = ax.imshow(heat_data.values, cmap='YlGn', vmin=0, vmax=1, aspect='auto')
            ax.set_xticks(range(n_dbs))
            ax.set_xticklabels(heat_data.columns, rotation=20, ha='right', fontsize=15)
            ax.set_yticks(range(n_models))
            ax.set_yticklabels(heat_data.index, fontsize=15)
            for _i in range(n_models):
                for _j in range(n_dbs):
                    v = heat_data.values[_i, _j]
                    if not np.isnan(v):
                        ax.text(_j, _i, f'{v:.3f}', ha='center', va='center',
                                color='white' if v > 0.5 else 'black', fontsize=15)
            plt.colorbar(im, ax=ax, label='PR-AUC')

        # ax.set_title('PR-AUC Heatmap', fontsize=14, fontweight='bold')
        ax.set_xlabel('Database', fontsize =15); ax.set_ylabel('Model', fontsize =15)
        plt.tight_layout()
        plt.show()

    # ── (3) 모델별 macro 평균 (1D) ────────────────────────────
    print()
    print("=" * 80)
    print("[PR-AUC] 모델별 macro 평균")
    print("=" * 80)
    macro_pr = reorder_models(pivot_prauc_view['mean'].dropna(), reverse=True)  # barh 위(제안모델)→아래

    fig, ax = plt.subplots(figsize=(10, max(4, 0.5*len(macro_pr))))
    colors = [label_color(l) for l in macro_pr.index]
    bars = ax.barh(macro_pr.index, macro_pr.values, color=colors,
                   edgecolor='#333', linewidth=0.8)
    ax.set_xlim(-0.02, 1.05)   # 0인 bar 도 보이도록
    ax.set_xlabel('PR-AUC (macro avg over DBs)', fontsize=11)
    ax.set_title('PR-AUC: 모델별 macro 평균 (DB 평균)',
                 fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    # 값 라벨
    for bar, val in zip(bars, macro_pr.values):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=10)
    plt.tight_layout()
    plt.show()

    # 표 형태로도 출력 (복사용)
    print(f"\n{macro_pr.sort_values(ascending=False).round(4).to_string()}")

In [ ]:
# === Recall@K + Recall@3K: Bar chart (2가지 버전) ===
import matplotlib.pyplot as plt
import numpy as np
# 모델별 명시적 색상 — 시각적 구분 명확화
# ColorBrewer Dark2 — 색약자 친화 + 흑백 인쇄에도 구분 가능
EXPLICIT_BAR_COLORS = {
    '제안모델':            '#1B9E77',  # teal-green
    'DeepJoin':            '#D95F02',  # orange
    'EmbDI':               '#7570B3',  # purple
    'COMA':                '#E7298A',  # pink
    'Cupid':               '#66A61E',  # green
    'DistributionBased':   '#E6AB02',  # gold
    'GPT':                 '#A6761D',  # brown
}

def bar_color(display_lbl):
    """명시적 매핑 우선 → 없으면 label_color() 로 fallback."""
    if display_lbl in EXPLICIT_BAR_COLORS:
        return EXPLICIT_BAR_COLORS[display_lbl]
    return label_color(display_lbl)

if df.empty:
    print("(no ranking data)")
else:
    # ── 공통 헬퍼: pivot 만들기 ─────────────────────────────────
    def make_pivot(metric):
        pv = df.pivot_table(index='label', columns='db', values=metric, aggfunc='first')
        pv['mean'] = pv.mean(axis=1, skipna=True)
        return pv.sort_values('mean', ascending=False)

    pivot_rk  = make_pivot('recall_at_K')
    pivot_r3k = make_pivot('recall_at_3K')

    # ── 인덱스(라벨)를 표시명으로 변환 (cell 3 미실행 대비 idempotent) ──
    if 'LABEL_DISPLAY_NAMES' in dir():
        pivot_rk.index  = [LABEL_DISPLAY_NAMES.get(l, l) for l in pivot_rk.index]
        pivot_r3k.index = [LABEL_DISPLAY_NAMES.get(l, l) for l in pivot_r3k.index]

    # 모델 표시 순서로 정렬 (heatmap/grouped bar 공통)
    pivot_rk  = reorder_models(pivot_rk)
    pivot_r3k = reorder_models(pivot_r3k)

    # ─────────────────────────────────────────────────────────────
    # (A0) Recall@K — Heatmap (PR-AUC 와 동일 형식)
    # ─────────────────────────────────────────────────────────────
    print("=" * 80); print("[Recall@K] 모델 × DB Heatmap"); print("=" * 80)
    _heat_rk = pivot_rk.drop(columns=['mean'], errors='ignore').copy()
    if 'LABEL_DISPLAY_NAMES' in dir():
        _heat_rk.index = [LABEL_DISPLAY_NAMES.get(l, l) for l in _heat_rk.index]
    _render_heatmap(_heat_rk, 'Recall@K')

    # ─────────────────────────────────────────────────────────────
    # (A) Recall@K — 모델 × DB grouped bar
    # ─────────────────────────────────────────────────────────────
    print("=" * 80)
    print("[Recall@K] 모델 × DB grouped bar")
    print("=" * 80)
    data_rk = pivot_rk.drop(columns=['mean'], errors='ignore')
    if not data_rk.empty:
        n_models, n_dbs = data_rk.shape
        # fig, ax = plt.subplots(figsize=(max(12, 1.8*n_dbs*n_models/3), 6))
        fig, ax = plt.subplots(figsize=(10, 6))
        data_rk.T.plot(kind='bar', ax=ax, width=0.55, edgecolor='#333', linewidth=0.5,
                       color=[bar_color(l) for l in data_rk.index])
        # ax.set_title('Recall@K (K=|GT|): model × DB',
                    #  fontsize=14, fontweight='bold')
        ax.set_ylabel('Recall@K', fontsize=15)
        ax.set_xlabel('Database', fontsize=15)
        ax.set_ylim(-0.03, 1.10)
        ax.grid(axis='y', alpha=0.2)
        ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0),
                  fontsize=12, title='Model', title_fontsize=12)
        plt.xticks(rotation=20, ha='right', fontsize=15)
        plt.yticks(fontsize=15)
        plt.tight_layout()
        plt.show()

    # ─────────────────────────────────────────────────────────────
    # (B) Recall@K — 모델별 macro 평균 bar
    # ─────────────────────────────────────────────────────────────
    print()
    print("=" * 80)
    print("[Recall@K] 모델별 macro 평균")
    print("=" * 80)
    macro_rk = reorder_models(pivot_rk['mean'].dropna(), reverse=True)  # barh 위(제안모델)→아래
    fig, ax = plt.subplots(figsize=(10, max(4, 0.5*len(macro_rk))))
    colors = [bar_color(l) for l in macro_rk.index]
    bars = ax.barh(macro_rk.index, macro_rk.values, color=colors,
                   edgecolor='#333', linewidth=0.8)
    ax.set_xlim(-0.02, 1.05)
    ax.set_xlabel('Recall@K (macro avg over DBs)', fontsize=11)
    ax.set_title('Recall@K: 모델별 macro 평균',
                 fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    for bar, val in zip(bars, macro_rk.values):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=10)
    plt.tight_layout()
    plt.show()
    print(f"\n{macro_rk.sort_values(ascending=False).round(4).to_string()}")

    # # ─────────────────────────────────────────────────────────────
    # # (C0) Recall@3K — Heatmap (PR-AUC 와 동일 형식)
    # # ─────────────────────────────────────────────────────────────
    # print("=" * 80); print("[Recall@3K] 모델 × DB Heatmap"); print("=" * 80)
    # _heat_r3k = pivot_r3k.drop(columns=['mean'], errors='ignore').copy()
    # if 'LABEL_DISPLAY_NAMES' in dir():
    #     _heat_r3k.index = [LABEL_DISPLAY_NAMES.get(l, l) for l in _heat_r3k.index]
    # _render_heatmap(_heat_r3k, 'Recall@3K')

    # # ─────────────────────────────────────────────────────────────
    # # (C) Recall@3K — 모델 × DB grouped bar
    # # ─────────────────────────────────────────────────────────────
    # print()
    # print("=" * 80)
    # print("[Recall@3K] 모델 × DB grouped bar")
    # print("=" * 80)
    # data_r3k = pivot_r3k.drop(columns=['mean'], errors='ignore')
    # if not data_r3k.empty:
    #     n_models, n_dbs = data_r3k.shape
    #     fig, ax = plt.subplots(figsize=(max(20, 1.8*n_dbs*n_models/3), 6))
    #     data_r3k.T.plot(kind='bar', ax=ax, width=0.55, edgecolor='#333', linewidth=0.5,
    #                     color=[bar_color(l) for l in data_r3k.index])
    #     ax.set_title('Recall@3K (3K = 3·|GT|): model × DB',
    #                  fontsize=14, fontweight='bold')
    #     ax.set_ylabel('Recall@3K', fontsize=11)
    #     ax.set_xlabel('Database', fontsize=11)
    #     ax.set_ylim(-0.03, 1.10)
    #     ax.grid(axis='y', alpha=0.3)
    #     ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0),
    #               fontsize=9, title='Model', title_fontsize=10)
    #     plt.xticks(rotation=30, ha='right')
    #     plt.tight_layout()
    #     plt.show()

    # # ─────────────────────────────────────────────────────────────
    # # (D) Recall@3K — 모델별 macro 평균 bar
    # # ─────────────────────────────────────────────────────────────
    # print()
    # print("=" * 80)
    # print("[Recall@3K] 모델별 macro 평균")
    # print("=" * 80)
    # macro_r3k = pivot_r3k['mean'].dropna().sort_values(ascending=True)
    # fig, ax = plt.subplots(figsize=(10, max(4, 0.5*len(macro_r3k))))
    # colors = [bar_color(l) for l in macro_r3k.index]
    # bars = ax.barh(macro_r3k.index, macro_r3k.values, color=colors,
    #                edgecolor='#333', linewidth=0.8)
    # ax.set_xlim(-0.02, 1.05)
    # ax.set_xlabel('Recall@3K (macro avg over DBs)', fontsize=11)
    # ax.set_title('Recall@3K: 모델별 macro 평균',
    #              fontsize=14, fontweight='bold')
    # ax.grid(axis='x', alpha=0.3)
    # for bar, val in zip(bars, macro_r3k.values):
    #     ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
    #             f'{val:.4f}', va='center', fontsize=10)
    # plt.tight_layout()
    # plt.show()
    # print(f"\n{macro_r3k.sort_values(ascending=False).round(4).to_string()}")

In [ ]:
# === PR 곡선 — DB 별 1개 차트, 각 차트 안에 모든 모델/config 별 라인 ===
# 5종 ranking JSON 의 y_true/y_score 를 통합하여 DB 별로 PR curve 비교
#
# 필요: 각 노트북의 ranking 셀에서 y_true/y_score 가 JSON 에 저장돼 있어야 함
#   - CRIM-ablation : cell 45 (7 configs)
#   - CRIM single   : cell 42
#   - Valentine     : cell 20 (3 matchers)
#   - DeepJoin      : cell 16
#   - EmbDI         : cell 13

from sklearn.metrics import precision_recall_curve, average_precision_score
import json as _json_pr
import os as _os_pr

# ── 1) 모든 ranking JSON 에서 (label, db, y_true, y_score) 수집 ──────
pr_data = {}  # data[db][label] = (y_true, y_score)

def _add_entry(db, label, m):
    yt = m.get('y_true'); ys = m.get('y_score')
    if yt is None or ys is None:
        return
    pr_data.setdefault(db, {})[label] = (np.array(yt), np.array(ys))

# (a) CRIM-ablation — CRIM_VERSION_FILTER 로 필터
_crim_ver = globals().get('CRIM_VERSION_FILTER', '')
for jp in sorted([_os_pr.path.join(str(SOURCE_DIR), f)
                  for f in _os_pr.listdir(str(SOURCE_DIR))
                  if f.startswith('ablation_ranking_metrics_')
                     and f.endswith('.json') and 'classic' not in f
                     and _crim_ver in f]):
    with open(jp) as f:
        d = _json_pr.load(f)
    for cfg, dbs in d.get('ranking_table', {}).items():
        for db, m in dbs.items():
            _add_entry(db, f'CRIM-ablation | {cfg}', m)

# (b) CRIM single — CRIM_VERSION_FILTER 로 필터
for jp in sorted([_os_pr.path.join(str(SOURCE_DIR), f)
                  for f in _os_pr.listdir(str(SOURCE_DIR))
                  if f.startswith('crim_ranking_metrics_') and f.endswith('.json')
                     and 'classic' not in f
                     and _crim_ver in f]):
    with open(jp) as f:
        d = _json_pr.load(f)
    db = d.get('db'); m = d
    if db:
        _add_entry(db, 'CRIM', m)

# (c) Valentine (3 matchers per db)
for jp in sorted([_os_pr.path.join(str(SOURCE_DIR), f)
                  for f in _os_pr.listdir(str(SOURCE_DIR))
                  if f.startswith('valentine_ranking_metrics_') and f.endswith('.json')
                     and 'classic' not in f]):
    with open(jp) as f:
        d = _json_pr.load(f)
    db = d.get('db')
    for matcher, m in d.get('per_model', {}).items():
        if db:
            _add_entry(db, f'Valentine-{matcher} | {matcher}', m)

# (d) DeepJoin
for jp in sorted([_os_pr.path.join(str(DEEPJOIN_DIR), f)
                  for f in _os_pr.listdir(str(DEEPJOIN_DIR))
                  if f.startswith('deepjoin_ranking_metrics') and f.endswith('.json')
                     and 'classic' not in f] if _os_pr.path.exists(str(DEEPJOIN_DIR)) else []):
    with open(jp) as f:
        d = _json_pr.load(f)
    for db, m in d.get('per_db', {}).items():
        _add_entry(db, 'DeepJoin', m)

# (e) EmbDI
for jp in sorted([_os_pr.path.join(str(SOURCE_DIR), f)
                  for f in _os_pr.listdir(str(SOURCE_DIR))
                  if f.startswith('embdi_ranking_metrics_') and f.endswith('.json')
                     and 'classic' not in f]):
    with open(jp) as f:
        d = _json_pr.load(f)
    db = d.get('db'); m = d
    if db:
        _add_entry(db, 'EmbDI', m)

# (f) GPT-4o
for jp in sorted([_os_pr.path.join(str(SOURCE_DIR), f)
                  for f in _os_pr.listdir(str(SOURCE_DIR))
                  if f.startswith('gpt4o_ranking_metrics_') and f.endswith('.json')
                     and 'classic' not in f]):
    with open(jp) as f:
        d = _json_pr.load(f)
    db = d.get('db'); m = d
    if db:
        _add_entry(db, 'GPT-4o', m)


# ── Legend 표시 순서 ─────────────────────────────────────────────
# 사용자 지정: 제안모델 -> Cupid -> COMA -> DistributionBased -> EmbDI -> DeepJoin.
# prefix 매칭으로 priority 부여. 매칭 안 되면 priority 99 (기타).
LABEL_ORDER_PREFIXES = [
    ('CRIM-ablation | stat+char+ov', 0),   # 제안모델 (Full ablation)
    ('CRIM', 0),                            # 제안모델 (single CRIM 파일)
    ('Valentine-CUPID', 1),                # 실제 JSON 의 matcher 이름 (대문자)
    ('Valentine-COMA',  2),
    ('Valentine-DIST',  3),                # DistributionBased — JSON 에선 'DIST' 약어
    ('EmbDI',           4),
    ('DeepJoin',        5),
    ('GPT-4o',          6),
]
def _label_priority(lbl):
    """prefix 매칭 — '|' 포함 prefix 는 정확 매칭, 나머지는 단어 경계 매칭."""
    for prefix, rank in LABEL_ORDER_PREFIXES:
        if '|' in prefix:
            # 'CRIM-ablation | stat+char+ov' 같은 fully-qualified label 은 정확 매칭만
            if lbl == prefix:
                return rank
        else:
            # 'CRIM', 'Valentine-CUPID' 같은 prefix 는 token boundary 매칭
            # ('CRIM' 이 'CRIM-ablation' 에 매칭되면 안 됨)
            if lbl == prefix or lbl.startswith(prefix + ' ') or lbl.startswith(prefix + '|'):
                return rank
    return 99   # 그 외 (다른 ablation config 등) — 맨 뒤

# ── 2) TARGET_DBS / LABEL_PRESETS 필터 적용 ────────────────────────
plot_dbs = sorted(set(pr_data.keys()) & set(TARGET_DBS))

if not plot_dbs:
    print("(PR curve 데이터 없음 — 각 모델 노트북의 ranking 셀을 재실행해서 "
          "y_true/y_score 가 JSON 에 저장되도록 하세요)")
else:
    # LABEL_PRESETS 필터
    allowed_labels = None
    if 'LABEL_PRESETS' in dir() and LABEL_PRESETS.get(LABEL_PRESET):
        allowed_labels = set(LABEL_PRESETS[LABEL_PRESET])

    # 사용 가능한 라벨 (LABEL_PRESET 미적용 시 전체)
    all_labels = set()
    for db_data in pr_data.values():
        all_labels.update(db_data.keys())
    if allowed_labels is not None:
        plot_labels = sorted(all_labels & allowed_labels)
    else:
        plot_labels = sorted(all_labels)

    if not plot_labels:
        print(f"⚠ LABEL_PRESET='{LABEL_PRESET}' 와 일치하는 PR curve 데이터 없음")
    else:
        # ── 3) 색상 매핑 ────────────────────────────────────────
        # 모델 그룹별로 색조 통일
        def _label_color(lbl, idx, total):
            if lbl.startswith('CRIM-ablation') and 'stat+char+ov' in lbl:
                return '#222222'        # full ablation → 검정
            if lbl.startswith('CRIM-ablation'):
                return plt.cm.Greens(0.35 + 0.5 * (idx / max(total, 1)))
            if lbl == 'CRIM':
                return '#8B0000'        # CRIM main → dark red
            if lbl == 'DeepJoin':
                return '#1f77b4'        # blue
            if lbl.startswith('Valentine'):
                return plt.cm.Oranges(0.5 + 0.3 * (idx / max(total, 1)))
            if lbl == 'EmbDI':
                return '#9467bd'        # purple
            if lbl == 'GPT-4o':
                return '#FF6B6B'        # coral red
            return plt.cm.tab20(idx % 20)

        def _label_style(lbl):
            if lbl.startswith('CRIM-ablation') and 'stat+char+ov' in lbl:
                return {'linewidth': 3.0, 'linestyle': '-', 'zorder': 10}
            if lbl == 'CRIM':
                return {'linewidth': 2.5, 'linestyle': '-', 'zorder': 9}
            if lbl in ('DeepJoin', 'EmbDI', 'GPT-4o'):
                return {'linewidth': 2.0, 'linestyle': '-', 'zorder': 8}
            if lbl.startswith('Valentine'):
                return {'linewidth': 1.5, 'linestyle': '--', 'zorder': 5}
            if lbl.startswith('CRIM-ablation'):  # single/2-branch
                return {'linewidth': 1.3, 'linestyle': ':', 'zorder': 3}
            return {'linewidth': 1.5, 'linestyle': '-'}

        # ── 4) 서브플롯 그리드 ─────────────────────────────────
        n_dbs = len(plot_dbs)
        n_cols = min(2, n_dbs)
        n_rows = (n_dbs + n_cols - 1) // n_cols

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(8*n_cols, 6*n_rows))
        if n_dbs == 1:
            axes_list = [axes]
        elif n_rows == 1 or n_cols == 1:
            axes_list = list(axes.flatten() if hasattr(axes, 'flatten') else axes)
        else:
            axes_list = list(axes.flatten())

        # DB 별 라벨 순서 기록용 (그래프 외부 텍스트 출력)
        db_label_order = {}

        for db_idx, db_name in enumerate(plot_dbs):
            ax = axes_list[db_idx]
            db_data = pr_data[db_name]

            # 라벨별 AP 계산
            lbl_aps = {}
            for lbl in plot_labels:
                if lbl not in db_data:
                    continue
                yt, ys = db_data[lbl]
                if yt.sum() == 0:
                    continue
                ap = average_precision_score(yt, ys)
                lbl_aps[lbl] = (yt, ys, ap)

            # 사용자 지정 순서 우선, 동일 priority 내에선 AP 내림차순
            sorted_labels = sorted(lbl_aps.keys(),
                                   key=lambda l: (_label_priority(l), -lbl_aps[l][2]))
            total_labels = len(sorted_labels)
            for idx, lbl in enumerate(sorted_labels):
                yt, ys, ap = lbl_aps[lbl]
                prec, rec, _ = precision_recall_curve(yt, ys)
                style = _label_style(lbl)
                # label 인자 제거 — 그래프에 legend 표시 안 함 (별도 텍스트 출력)
                ax.plot(rec, prec, color=label_color(lbl), **style)
            # DB 별 라벨 순서 기록 (그래프 외부 텍스트 출력용)
            db_label_order[db_name] = sorted_labels

            # # Random baseline
            # if lbl_aps:
            #     sample_yt = next(iter(lbl_aps.values()))[0]
            #     baseline = sample_yt.sum() / len(sample_yt)
            #     ax.axhline(baseline, color='gray', linestyle=':', alpha=0.5,
            #                label=f'random (P={baseline:.3f})', linewidth=1)

            ax.set_xlabel('Recall', fontsize=11)
            ax.set_ylabel('Precision', fontsize=11)
            ax.set_title(f'PR Curve — {db_name}', fontsize=13, fontweight='bold')
            ax.set_xlim(0, 1.02)
            ax.set_ylim(-0.03, 1.05)   # 하한 음수로 → y=0 근처 라인도 보이게
            # ★ 범례 제거 — 그래프 밖에서 별도 텍스트로 출력
            ax.grid(alpha=0.3)

        # 남는 subplot 숨김
        for idx in range(n_dbs, len(axes_list)):
            axes_list[idx].set_visible(False)

        plt.suptitle('Precision-Recall Curves per DB — all models',
                     fontsize=14, fontweight='bold', y=1.005)
        plt.tight_layout()
        plt.show()

        # ── 라벨 (범례) — 별도 figure 로 PR curve 외부 출력 ────────
        # 모든 DB 의 라벨 union → priority 순 정렬 (그래프 라인 순서와 동일)
        import matplotlib.lines as _mlines
        _all_used = set()
        for _labels in db_label_order.values():
            _all_used.update(_labels)
        _sorted_union = sorted(_all_used, key=lambda l: _label_priority(l))

        # 각 라벨의 색/스타일로 Line2D handle 생성
        _handles = []
        for _lbl in _sorted_union:
            _style = _label_style(_lbl)
            _color = label_color(_lbl)
            # zorder 같은 비표시 인자는 Line2D 에서 무시 안 함 — 그대로 전달
            _h = _mlines.Line2D([], [], color=_color, label=display_label(_lbl), **_style)
            _handles.append(_h)

        # 별도 legend-only figure (흰 배경, 박스)
        _fig_leg, _ax_leg = plt.subplots(figsize=(5, max(1.5, 0.4 * len(_handles))))
        _ax_leg.axis('off')
        _ax_leg.legend(handles=_handles, loc='center',
                       frameon=True, framealpha=1.0, fancybox=True,
                       fontsize=11, ncol=1, title='Models',
                       title_fontsize=12)
        plt.tight_layout()
        plt.show()

        # ── 요약 표 (DB × label 의 AP) ──────────────────────
        print()
        print("=" * 80)
        print("[PR Curve AP (= PR-AUC) 요약: DB × label]")
        print("=" * 80)
        ap_table = pd.DataFrame(index=plot_labels, columns=plot_dbs, dtype=float)
        for db in plot_dbs:
            for lbl in plot_labels:
                if lbl in pr_data[db]:
                    yt, ys = pr_data[db][lbl]
                    if yt.sum() > 0:
                        ap_table.loc[lbl, db] = average_precision_score(yt, ys)
        ap_table['mean'] = ap_table.mean(axis=1, skipna=True)
        # 사용자 지정 순서 적용 (원본 label 기준으로 priority 계산 후 정렬)
        ap_table['_order'] = [_label_priority(l) for l in ap_table.index]
        ap_table = ap_table.sort_values(['_order', 'mean'], ascending=[True, False])
        ap_table = ap_table.drop(columns='_order')
        # 정렬 후 인덱스를 표시명으로 변환
        if 'LABEL_DISPLAY_NAMES' in dir():
            ap_table.index = [LABEL_DISPLAY_NAMES.get(l, l) for l in ap_table.index]
        display(ap_table.style.format('{:.4f}', na_rep='N/A')
                .background_gradient(cmap='YlGn', axis=None, low=0.0, high=1.0))


In [ ]:
# === 평균 MRR — 모델별 Bar Chart (MRR + 1/MRR) ===
# 1/MRR = 첫 번째 정답이 등장하는 평균 rank 위치 (낮을수록 좋음)

import matplotlib.pyplot as plt
import matplotlib.colors as _mcolors

def compute_mrr(y_true, y_score):
    yt = np.asarray(y_true); ys = np.asarray(y_score)
    if yt.sum() == 0:
        return 0.0
    order = np.argsort(-ys, kind='stable')
    ranked_truth = yt[order]
    rrs = [1.0 / (i + 1) for i, v in enumerate(ranked_truth) if v == 1]
    return float(np.mean(rrs)) if rrs else 0.0


if 'pr_data' not in dir() or not pr_data:
    print('⚠ pr_data 가 없습니다. cell 24 (PR curve) 를 먼저 실행하세요.')
else:
    # ── DB × label MRR 표 ─────────────────────────────────────
    mrr_table = pd.DataFrame(index=plot_labels, columns=plot_dbs, dtype=float)
    for db in plot_dbs:
        for lbl in plot_labels:
            if lbl in pr_data[db]:
                yt, ys = pr_data[db][lbl]
                mrr_table.loc[lbl, db] = compute_mrr(yt, ys)

    mrr_table['mean'] = mrr_table.mean(axis=1, skipna=True)

    # PR curve legend 와 동일한 순서 — globals 의존 X, 항상 최신 prefix 사용
    _LABEL_ORDER_PREFIXES = [
        ('CRIM-ablation | stat+char+ov', 0),
        ('CRIM', 0),
        ('Valentine-CUPID', 1),
        ('Valentine-COMA',  2),
        ('Valentine-DIST',  3),
        ('EmbDI',           4),
        ('DeepJoin',        5),
        ('GPT-4o',          6),
    ]
    def _label_priority_local(lbl):
        for prefix, rank in _LABEL_ORDER_PREFIXES:
            if '|' in prefix:
                if lbl == prefix:
                    return rank
            else:
                if lbl == prefix or lbl.startswith(prefix + ' ') or lbl.startswith(prefix + '|'):
                    return rank
        return 99

    # 정렬 + 진단 출력
    mrr_table['_order'] = [_label_priority_local(l) for l in mrr_table.index]
    mrr_table = mrr_table.sort_values('_order', kind='stable')

    print('[MRR sort 진단] priority 매칭 상태:')
    for l, o in zip(mrr_table.index, mrr_table['_order']):
        marker = '⚠' if o == 99 else ' '
        print(f'  {marker} priority={o:>2}  {l}')

    mrr_table = mrr_table.drop(columns='_order')

    # 모델별 macro 평균 (priority 순서 유지)
    macro_mrr = mrr_table['mean'].dropna()

    # display name 변환 — globals 우선, fallback 으로 inline mapping
    _LDN = globals().get('LABEL_DISPLAY_NAMES', {
        'CRIM-ablation | stat+char+ov': '제안모델',
        'CRIM':                         '제안모델',
        'Valentine-CUPID | CUPID':      'Cupid',
        'Valentine-COMA | COMA':        'COMA',
        'Valentine-DIST | DIST':        'DistributionBased',
        'EmbDI':                        'EmbDI',
        'DeepJoin':                     'DeepJoin',
        'GPT-4o':                       'GPT',
        'GPT-4o | GPT-4o':              'GPT',
    })
    macro_mrr.index = [_LDN.get(l, l) for l in macro_mrr.index]

    # 1/MRR 계산 (= 평균 첫 정답 rank)
    inv_mrr = 1.0 / macro_mrr.replace(0, np.nan)

    # ── 색상 매핑 (bar chart cell 23 의 EXPLICIT_BAR_COLORS 재사용) ──
    if 'EXPLICIT_BAR_COLORS' in dir():
        _color_map = EXPLICIT_BAR_COLORS
    else:
        _color_map = {}
    def _bar_color(lbl):
        return _color_map.get(lbl, '#888888')

    # ── (A0) MRR — Heatmap (PR-AUC 와 동일 형식) ────────────
    print("=" * 80); print("[MRR] 모델 × DB Heatmap"); print("=" * 80)
    _heat_mrr = mrr_table.drop(columns=['mean'], errors='ignore').copy()
    if 'LABEL_DISPLAY_NAMES' in dir():
        _heat_mrr.index = [LABEL_DISPLAY_NAMES.get(l, l) for l in _heat_mrr.index]
    _render_heatmap(_heat_mrr, 'MRR')

    # ── (A) MRR 막대 차트 ────────────────────────────────────
    FS = 20   # 폰트사이즈

    fig, ax = plt.subplots(figsize=(max(10, 1.2 * len(macro_mrr)), 6))
    colors = [_bar_color(l) for l in macro_mrr.index]
    bars = ax.bar(macro_mrr.index, macro_mrr.values, color=colors,
                  edgecolor='#333', linewidth=0.8)

    # 값 라벨 (MRR + 1/MRR)
    for bar, lbl, mrr_val in zip(bars, macro_mrr.index, macro_mrr.values):
        height = bar.get_height()
        inv = inv_mrr.get(lbl, np.nan)
        label_text = f'{mrr_val:.4f}\n(1/MRR={inv:.2f})' if not np.isnan(inv) else f'{mrr_val:.4f}'
        ax.text(bar.get_x() + bar.get_width()/2, height + 0.015,
                label_text, ha='center', va='bottom', fontsize=FS - 4)

    ax.set_ylim(0, max(1.05, macro_mrr.max() * 1.20))
    # ax.set_ylabel('MRR (macro avg over DBs)', fontsize=FS)
    ax.set_ylabel('MRR', fontsize=FS)
    ax.set_xlabel('Model', fontsize=FS)
    # ax.set_title('MRR — 모델별 평균 (1/MRR 함께 표시)',
    #              fontsize=FS + 2, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=20, ha='right', fontsize=FS)
    plt.yticks(fontsize=FS)
    plt.tight_layout()
    plt.show()

    # ── 표 형태로도 출력 ─────────────────────────────────────
    print()
    print('=' * 60)
    print('[MRR 모델별 macro 평균 + 1/MRR]')
    print('=' * 60)
    summary_df = pd.DataFrame({
        'MRR':   macro_mrr.round(4),
        '1/MRR': inv_mrr.round(2),
    })
    display(summary_df.style.format({'MRR': '{:.4f}', '1/MRR': '{:.2f}'}))


## 🆔 Canonical Identifier Protocol — 평가 일관성 보장

평가 단계에서 발생할 수 있는 **schema-leakage** 를 차단하기 위해 모든 컬럼에
schema-independent canonical identifier `(T_i, C_j)` 를 부여:

- `T_i` : 테이블 인덱스 (원본 알파벳 순)
- `C_j` : 테이블 내 DataFrame 컬럼 위치 인덱스

**효과**:
1. 원본 / 마스크 폴더 모두 **같은 canonical ID 공간** 으로 사상됨
2. 같은 logical column 은 폴더 무관 항상 같은 `(T_i, C_j)` 부여
3. GT, 예측, TP/FP/FN 계산이 모두 canonical 공간에서 일관 수행
4. 컬럼명 차이로 인한 **evaluation-side variation 제거**

**남는 차이**: 모델 inference 내부에서 컬럼명을 사용하는 경우 (예: Valentine
DistributionBased) 의 score 변동은 여전히 존재 — 이는 **algorithm-level
schema leakage** 로 별도 보고.

In [ ]:
# === Canonical Identifier 구축 + Evaluation 일관성 검증 ===
import pathlib as _pathlib_can
import pandas as _pd_can

BIRD_ROOT = _pathlib_can.Path('../workspace/dataset_mini/bird_data_tables')

def build_canonical_map(db_folder: _pathlib_can.Path):
    """
    DB 폴더의 모든 컬럼에 canonical ID (T_i, C_j) 부여.
    - 마스크 폴더는 mask_mapping.json 으로 원본 위치 추적
    - 원본 폴더는 직접 알파벳 순 + DataFrame 컬럼 위치 사용
    """
    csv_files = sorted(db_folder.glob('*.csv'))

    # mask_mapping 있으면 mask → orig 역매핑 구축
    mm_path = db_folder / 'mask_mapping.json'
    if mm_path.exists():
        with open(mm_path) as f:
            mm = _json_pr.load(f)
        # tuple-of-lists 또는 dict 형식 모두 지원
        tm_pairs = mm.get('table_mapping', [])
        if isinstance(tm_pairs, dict):
            tm_pairs = list(tm_pairs.items())
        mask2orig_table = {m: o for o, m in tm_pairs}
        cm = mm.get('col_mapping', [])
        if isinstance(cm, dict):
            cm = list(cm.items())
        mask2orig_col = {}
        for orig_t, col_dict in cm:
            mask_t = next((m for o, m in tm_pairs if o == orig_t), orig_t)
            for orig_c, mask_c in col_dict.items():
                mask2orig_col[(mask_t, mask_c)] = orig_c
    else:
        mask2orig_table = {}
        mask2orig_col = {}

    # logical table 순서 결정 (원본 이름 알파벳 순)
    file_records = []
    for csv_path in csv_files:
        mask_name = csv_path.stem
        orig_name = mask2orig_table.get(mask_name, mask_name)
        file_records.append((orig_name, mask_name, csv_path))
    file_records.sort(key=lambda x: x[0])    # ← 원본 이름 기준 정렬

    canonical = {}     # (현재 table_name, 현재 col_name) → (T_i, C_j)
    logical_meta = []  # (T_i, orig_name, mask_name, [(orig_col, mask_col), ...])

    for t_idx, (orig_t, mask_t, csv_path) in enumerate(file_records):
        df = _pd_can.read_csv(csv_path, encoding='utf-8-sig',
                              on_bad_lines='skip', low_memory=False)
        cols_meta = []
        for c_idx, col in enumerate(df.columns):
            # 현재 컬럼명 → canonical
            canonical[(mask_t, col)] = (f"T{t_idx:02d}", f"C{c_idx:02d}")
            orig_col = mask2orig_col.get((mask_t, col), col)
            cols_meta.append((orig_col, col))
        logical_meta.append((t_idx, orig_t, mask_t, cols_meta))

    return canonical, logical_meta


# ─── TARGET_DBS 각 DB 의 canonical map 구축 ──────────────────────
canonical_maps = {}    # {db_name: {(table, col): (T_i, C_j)}}
logical_metas  = {}    # {db_name: [(t_idx, orig_t, mask_t, cols), ...]}

for db in TARGET_DBS:
    folder = BIRD_ROOT / db
    if not folder.exists():
        print(f"⚠ DB 폴더 없음: {db}")
        continue
    try:
        cmap, lmeta = build_canonical_map(folder)
        canonical_maps[db] = cmap
        logical_metas[db]  = lmeta
        # 마스크 여부 표기
        is_mask = (folder / 'mask_mapping.json').exists()
        tag = '🔒 masked' if is_mask else '📝 original'
        print(f"  ✓ {db:<30} [{tag}] → {len(cmap)}개 컬럼")
    except Exception as e:
        print(f"  ✗ {db}: {e}")

print(f"\n총 {len(canonical_maps)}개 DB 의 canonical mapping 구축 완료")

In [ ]:
# === Canonical 공간에서 GT 비교 — orig vs mask 가 같은 GT 인지 검증 ===
import json as _json_chk

def load_gt_canonical(db_folder, canonical_map):
    """ground_truth.json 을 canonical ID 공간의 frozenset 으로 변환."""
    gt_path = db_folder / 'ground_truth.json'
    if not gt_path.exists():
        return None
    with open(gt_path) as f:
        gt_list = _json_chk.load(f)
    gt_canonical = set()
    for item in gt_list:
        ta, ca = item[0].split('.')
        tb, cb = item[1].split('.')
        cid_a = canonical_map.get((ta, ca))
        cid_b = canonical_map.get((tb, cb))
        if cid_a and cid_b:
            gt_canonical.add(frozenset({cid_a, cid_b}))
    return gt_canonical


# ─── (orig, mask) DB 쌍에 대해 canonical GT 가 동일한지 검증 ───
print("=" * 70)
print("[검증 1] orig vs mask DB 의 canonical GT 일치 여부")
print("=" * 70)
db_bases = sorted({db.replace('_mask', '') for db in TARGET_DBS})
gt_canonical_per_db = {}

for base in db_bases:
    orig_db = base
    mask_db = f'{base}_mask'

    gt_orig = None; gt_mask = None
    if orig_db in canonical_maps:
        gt_orig = load_gt_canonical(BIRD_ROOT / orig_db, canonical_maps[orig_db])
        gt_canonical_per_db[orig_db] = gt_orig
    if mask_db in canonical_maps:
        gt_mask = load_gt_canonical(BIRD_ROOT / mask_db, canonical_maps[mask_db])
        gt_canonical_per_db[mask_db] = gt_mask

    print(f"\n  📊 {base}")
    if gt_orig is not None:
        print(f"    orig GT (canonical)  : {len(gt_orig)} pairs")
    if gt_mask is not None:
        print(f"    mask GT (canonical)  : {len(gt_mask)} pairs")
    if gt_orig is not None and gt_mask is not None:
        match = (gt_orig == gt_mask)
        flag = '✅ 일치' if match else '❌ 불일치'
        print(f"    → orig ≡ mask        : {flag}")
        if not match:
            only_orig = gt_orig - gt_mask
            only_mask = gt_mask - gt_orig
            print(f"      orig 에만 있음: {len(only_orig)}, mask 에만 있음: {len(only_mask)}")

In [ ]:
# === Canonical 공간에서 모델 출력 (y_true) 일관성 검증 ===
# 같은 logical DB 의 orig 와 mask 가 같은 모델로 평가될 때
# y_true 배열의 분포 (특히 |positives|) 가 일치해야 함

print("=" * 70)
print("[검증 2] 각 모델의 orig vs mask y_true 분포 일치 여부")
print("=" * 70)

# pr_data 는 PR curve cell 에서 만들어진 것 활용
if 'pr_data' not in dir() or not pr_data:
    print("⚠ pr_data 가 없습니다. PR curve cell (마지막) 을 먼저 실행하세요.")
else:
    issue_count = 0
    for base in db_bases:
        orig_db = base
        mask_db = f'{base}_mask'

        if orig_db not in pr_data and mask_db not in pr_data:
            continue

        # 두 DB 의 라벨 교집합
        labels_orig = set(pr_data.get(orig_db, {}).keys())
        labels_mask = set(pr_data.get(mask_db, {}).keys())
        common_labels = labels_orig & labels_mask

        for lbl in sorted(common_labels):
            yt_o, ys_o = pr_data[orig_db][lbl]
            yt_m, ys_m = pr_data[mask_db][lbl]

            len_match = (len(yt_o) == len(yt_m))
            pos_match = (yt_o.sum() == yt_m.sum())

            if not (len_match and pos_match):
                issue_count += 1
                print(f"  ❌ {base} | {lbl}")
                print(f"     orig: len={len(yt_o)}, |pos|={int(yt_o.sum())}")
                print(f"     mask: len={len(yt_m)}, |pos|={int(yt_m.sum())}")

    if issue_count == 0:
        print(f"  ✅ 모든 (model × db) 쌍에서 |y_true| 와 |positives| 일치")
        print(f"     → evaluation 은 canonical-equivalent 한 상태")
    else:
        print(f"\n  ⚠ {issue_count}개 쌍에서 불일치 발견 — 진단 필요")

In [ ]:
# === Canonical 공간에서 직접 metric 재계산 (orig vs mask 비교) ===
# 같은 logical DB 의 orig/mask 결과를 canonical 공간에서 옆에 두고 비교

from sklearn.metrics import average_precision_score as _ap_can

print("=" * 90)
print("[Canonical 공간 비교] 같은 logical DB 의 orig vs mask 결과 (모델별)")
print("=" * 90)

if 'pr_data' in dir() and pr_data:
    rows = []
    for base in db_bases:
        orig_db = base
        mask_db = f'{base}_mask'
        if orig_db not in pr_data and mask_db not in pr_data:
            continue
        labels_o = set(pr_data.get(orig_db, {}).keys())
        labels_m = set(pr_data.get(mask_db, {}).keys())
        common = sorted(labels_o & labels_m)

        for lbl in common:
            yt_o, ys_o = pr_data[orig_db][lbl]
            yt_m, ys_m = pr_data[mask_db][lbl]
            if yt_o.sum() == 0 or yt_m.sum() == 0:
                continue
            ap_o = _ap_can(yt_o, ys_o)
            ap_m = _ap_can(yt_m, ys_m)

            # 표시 라벨
            disp = LABEL_DISPLAY_NAMES.get(lbl, lbl) if 'LABEL_DISPLAY_NAMES' in dir() else lbl

            rows.append({
                'DB (canonical)': base,
                'Model':          disp,
                'AP_orig':        round(ap_o, 4),
                'AP_mask':        round(ap_m, 4),
                'Δ (mask-orig)':  round(ap_m - ap_o, 4),
                '|GT|':           int(yt_o.sum()),
                '|candidates|':   len(yt_o),
            })

    if rows:
        comp_df = pd.DataFrame(rows)
        comp_df = comp_df.sort_values(['DB (canonical)', 'AP_orig'], ascending=[True, False])
        display(comp_df.style
                .format({'AP_orig': '{:.4f}', 'AP_mask': '{:.4f}',
                         'Δ (mask-orig)': '{:+.4f}'})
                .background_gradient(subset=['Δ (mask-orig)'],
                                      cmap='RdBu_r', vmin=-0.1, vmax=0.1)
                .set_caption('Canonical 공간 비교 — Δ 가 0 에 가까우면 schema-invariant'))

        # 모델별 평균 |Δ| 요약
        print("\n=== 모델별 |Δ AP| 평균 (낮을수록 schema-invariant) ===")
        summary = comp_df.copy()
        summary['|Δ|'] = summary['Δ (mask-orig)'].abs()
        ranking = summary.groupby('Model')['|Δ|'].mean().sort_values()
        for model, mean_abs in ranking.items():
            print(f"  {model:<25} {mean_abs:.4f}")
    else:
        print("  (비교할 (orig, mask) 쌍이 없습니다)")
else:
    print("⚠ pr_data 가 없음 — PR curve cell 실행 필요")